# Camada Bronze: Ingestão e Persistência de Dados Brutos

Execução da camada de ingestão do modelo medalhão para absorção dos arquivos operacionais da V-Credit. Em conformidade com as diretrizes de engenharia de dados, os registros são persistidos em formato Delta sem aplicação prévia de regras de negócio ou transformações complexas, mantendo a fidelidade estrutural dos dados originais. Adicionalmente, injetam-se metadados de auditoria temporal (`ingestion_timestamp`) e assegura-se a idempotência via modo de gravação `overwrite`.

In [0]:
%run ./00_Config

## Mapeamento de Fontes e Ingestão Automatizada

Configuração parametrizada dos arquivos de origem e seus respectivos parâmetros de codificação. Para bases legadas contendo caracteres acentuados (`Chamados_Hora.CSV` e `clientes.csv`), aplica-se explicitamente a codificação `windows-1252` para prevenir corrupção de dados textuais.

In [0]:
from pyspark.sql.functions import current_timestamp

# Dicionário mapeando o nome da tabela Delta de destino, o arquivo de origem e opções específicas (como encoding)
data_sources = {
    "tb_base_atendentes": {"path": f"{landing_path}/base_atendentes.csv", "encoding": None},
    "tb_base_motivos": {"path": f"{landing_path}/base_motivos.csv", "encoding": None},
    "tb_canais": {"path": f"{landing_path}/canais.csv", "encoding": None},
    "tb_chamados": {"path": f"{landing_path}/chamados.csv", "encoding": None},
    "tb_chamados_hora": {"path": f"{landing_path}/Chamados_Hora.CSV", "encoding": "windows-1252"},
    "tb_clientes": {"path": f"{landing_path}/clientes.csv", "encoding": "windows-1252"},
    "tb_custos": {"path": f"{landing_path}/custos.csv", "encoding": None},
    "tb_pesquisa_satisfacao": {"path": f"{landing_path}/pesquisa_satisfacao.csv", "encoding": None}
}

# Dicionário auxiliar para capturar os DataFrames e exibir o sumário posteriormente
dfs_loaded = {}

for table_name, config_info in data_sources.items():
    file_path = config_info["path"]
    file_encoding = config_info["encoding"]
    
    # Leitura condicional aplicando o encoding quando necessário
    reader = spark.read.option("header", "true")
    if file_encoding:
        reader = reader.option("encoding", file_encoding)
        
    df_raw = reader.csv(file_path)
    dfs_loaded[table_name] = df_raw
    
    # Gravação idempotente com metadados de auditoria e atualização de schema
    df_raw \
        .withColumn("ingestion_timestamp", current_timestamp()) \
        .write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{bronze_schema}.{table_name}")

print("[SUCESSO] Ingestão da camada Bronze concluída idempotentemente para todas as fontes.")

## Auditoria e Validação Volumétrica da Carga

Geração de sumário consolidado com a contagem de registros por tabela Delta persistida, permitindo auditar o sucesso do processamento e verificar a volumetria dos dados ingestados.

In [0]:
load_summary = [
    ("base_atendentes.csv", dfs_loaded["tb_base_atendentes"].count()),
    ("base_motivos.csv", dfs_loaded["tb_base_motivos"].count()),
    ("canais.csv", dfs_loaded["tb_canais"].count()),
    ("chamados.csv", dfs_loaded["tb_chamados"].count()),
    ("Chamados_Hora.CSV", dfs_loaded["tb_chamados_hora"].count()),
    ("clientes.csv", dfs_loaded["tb_clientes"].count()),
    ("custos.csv", dfs_loaded["tb_custos"].count()),
    ("pesquisa_satisfacao.csv", dfs_loaded["tb_pesquisa_satisfacao"].count()),
]

df_load_summary = spark.createDataFrame(
    load_summary,
    ["arquivo", "quantidade_registros"]
    
)

display(df_load_summary.orderBy("arquivo"))